In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import wandb
from scipy.stats import gmean

In [ ]:
# Results come from Weights & Biases. Point these at the project the
# experiments were logged to; the server address and credentials come from the
# usual wandb configuration (the WANDB_BASE_URL environment variable or
# ~/.config/wandb/settings).
WANDB_PROJECT = "fair-irl"
WANDB_ENTITY = None  # None uses the default entity of your wandb login

# Every run of one execution of Fair_IRL_Biased_Demonstrations.py shares a
# SESSION_ID. Only one session is ever loaded, so a plot can never mix results
# from two executions of the training script. Leave this None to use the most
# recent execution, or set it to a session id (e.g. "20260826-010125-3f9ab2")
# to reproduce an earlier set of figures exactly.
WANDB_SESSION = None


def to_tuple(data):
    if isinstance(data, list):
        return tuple(to_tuple(i) for i in data)
    elif isinstance(data, dict):
        return {k: to_tuple(v) for k, v in data.items()}
    else:
        return data


class ResultsLookup(dict):
    """Nested results dict that names the missing configuration on a miss.

    The plotting cells below index five levels deep, e.g.
    `averaged_data[dataset][expert][algorithm][dataset_bias_type][weight_adjust]`.
    With a plain defaultdict a configuration the session never ran would
    silently produce an empty frame, and the failure only surfaced later as a
    confusing
    `KeyError: 'sum_abs_subdominance_val'` from `.loc`. This reports which
    configuration is actually absent, and what the session does contain.
    """

    LEVELS = ("dataset", "expert", "algorithm", "dataset_bias_type", "weight_adjust")

    def __init__(self, depth=0, path=(), session_id=""):
        super().__init__()
        self._depth = depth
        self._path = path
        self._session_id = session_id

    def child(self, key):
        """Get or create the sub-lookup for `key` (never triggers __missing__)."""
        return self.setdefault(
            key,
            ResultsLookup(self._depth + 1, self._path + (key,), self._session_id),
        )

    def __missing__(self, key):
        level = self.LEVELS[self._depth]
        context = ", ".join(f"{n}={v!r}" for n, v in zip(self.LEVELS, self._path))
        raise KeyError(
            f"No results for {level}={key!r}"
            + (f" under {context}" if context else "")
            + f" in W&B session {self._session_id!r}."
            f" Available {level} values there: {sorted(self.keys(), key=repr)}."
            " Either re-run the training script with a config that produces it,"
            " or set WANDB_SESSION to a session that has it."
        )


def resolve_session(api, path, session=None):
    """Return the session id to load: `session`, or the most recent one."""
    if session is not None:
        return session

    newest = next(iter(api.runs(path, order="-created_at", per_page=1)), None)
    if newest is None:
        raise RuntimeError(
            f"No runs found in W&B project {path!r}."
            " Run Fair_IRL_Biased_Demonstrations.py first."
        )
    return newest.config["SESSION_ID"]


def read_session_runs(project, entity=None, session=None):
    """Read the runs of a single execution of the training script.

    Each run is one trial of one algorithm, one dataset bias type and one
    weight adjustment. Its config holds the experiment parameters and its
    summary holds that trial's results.

    Returns
    -------
    session_id : str
        The session that was loaded.
    records : list<dict>
        One record per run, with its `created_at`, `config` and `metrics`.
    """
    api = wandb.Api()
    path = f"{entity}/{project}" if entity else project
    session_id = resolve_session(api, path, session)

    records = []
    for run in api.runs(
        path, filters={"state": "finished", "config.SESSION_ID": session_id}
    ):
        summary = dict(run.summary)
        # Trials that did not converge have no results to plot.
        if not summary.get("converged", False):
            continue

        # Config values come back from W&B as lists; the plotting cells index
        # with tuple literals, so they have to be tuples to be usable as keys.
        config = to_tuple(
            {k: v for k, v in run.config.items() if not k.startswith("_")}
        )
        metrics = {
            k: v
            for k, v in summary.items()
            if not k.startswith("_")
            and isinstance(v, (int, float))
            and not isinstance(v, bool)
        }

        records.append(
            {"created_at": run.created_at, "config": config, "metrics": metrics}
        )

    return session_id, records


session_id, records = read_session_runs(WANDB_PROJECT, WANDB_ENTITY, WANDB_SESSION)
if not records:
    raise RuntimeError(
        f"W&B session {session_id!r} has no finished, converged runs."
        " Pick another session with WANDB_SESSION, or re-run the training script."
    )

# Collect the trials of each (dataset, expert, algorithm, dataset bias type,
# weight adjustment) configuration within this one session. Each run covers
# exactly one algorithm, one bias type and one weight adjustment, so the latter
# two are single tuples -- `()` for the unbiased dataset and for the unadjusted
# weights. Sessions logged before the Superhuman Fairness baseline existed have
# no ALGORITHM in their config; those runs are all FairIRL Bias Reduction ones.
trials_by_config = defaultdict(list)
config_by_key = {}
first_seen = {}
for record in records:
    config = record["config"]
    config_key = (
        config["DATASET"],
        config["EXPERT_ALGO"],
        config.get("ALGORITHM", "FairIRL Bias Reduction"),
        config["DATASET_BIAS_TYPE"],
        config["WEIGHT_ADJUST"],
    )
    trials_by_config[config_key].append(record["metrics"])
    config_by_key[config_key] = config
    first_seen[config_key] = min(
        first_seen.get(config_key, record["created_at"]), record["created_at"]
    )

# Average across all trials of each configuration
averaged_data = ResultsLookup(session_id=session_id)
averaged_info = {}
# Oldest configuration first, so that datasets appear in the order they were
# run, which is the order the plots lay them out along the x-axis.
for config_key in sorted(trials_by_config, key=lambda k: first_seen[k]):
    dataset, expert, algorithm, dataset_bias_type, weight_adjust = config_key
    trial_data = pd.DataFrame(trials_by_config[config_key])
    averaged_data.child(dataset).child(expert).child(algorithm).child(
        dataset_bias_type
    )[weight_adjust] = trial_data.mean()
    averaged_info.setdefault(dataset, {}).setdefault(expert, {}).setdefault(
        algorithm, {}
    ).setdefault(dataset_bias_type, {})[weight_adjust] = config_by_key[config_key]

print(f"W&B session:  {session_id}")
print(f"runs loaded:  {len(records)} across {len(trials_by_config)} configurations")
for config_key in sorted(trials_by_config, key=lambda k: first_seen[k]):
    dataset, expert, algorithm, dataset_bias_type, weight_adjust = config_key
    n = len(trials_by_config[config_key])
    print(
        f"  {dataset} / {expert} / {algorithm} / bias={dataset_bias_type}"
        f" / weights={weight_adjust}: {n} trial(s)"
    )

# Keep the notebook namespace clean for the plotting cells below. Done by
# name so that it stays correct however many of these were actually bound.
for _name in (
    "records",
    "record",
    "config",
    "trials_by_config",
    "config_by_key",
    "first_seen",
    "config_key",
    "trial_data",
    "dataset",
    "expert",
    "algorithm",
    "dataset_bias_type",
    "weight_adjust",
):
    globals().pop(_name, None)
del _name

In [ ]:
selected_expert = "OptClfMDPPol"

# Which technique the plots below are drawn for. Every run carries its
# technique as `ALGORITHM`, so switching this re-draws the same figures for
# another one; a comparison plot can index several, e.g.
# `averaged_data[dataset][selected_expert]["Superhuman Fairness"][bias][()]`.
# Valid values are the entries of the training script's ALGORITHMS list:
# "FairIRL Bias Reduction", "Superhuman Fairness", "Post Proc DP",
# "Post Proc EqOdds", "Fair LogLoss DP" and "Fair LogLoss EqOdds".
selected_algorithm = "FairIRL Bias Reduction"

# Each W&B run covers exactly one dataset bias type and one weight adjustment,
# so every key below is a single tuple, matching one entry of the training
# script's DATASET_BIAS_TYPE_LIST / WEIGHT_ADJUST_LIST -- plus the `()` that is
# always run for the unbiased dataset and for the unadjusted weights. Only
# configurations the loaded session actually ran can be selected here; every
# technique other than FairIRL Bias Reduction has no reward weights to adjust,
# so those only ever report the unadjusted `()` one.
unbiased_types = [
    (),
]
biased_types = [
    # ("unbalanced_redlining", 0.2),
    ("balanced_redlining", 0.2),
    # ("perfectly_balanced_redlining", 0.2),
    ("corruption_bias", "CatBoost", 0.001, "gaussian", 1.0),
]

unadjusted_weights = ()
adjusted_weights = (
    # ("mul_negative_weights", 0.0),
    # ("mul_negative_weights", 0.1),
    # ("mul_negative_weights", 0.2),
    # ("mul_negative_weights", 0.3),
    # ("mul_negative_weights", 0.4),
    # ("mul_negative_weights", 0.5),
    # ("mul_negative_weights", 0.6),
    # ("mul_negative_weights", 0.7),
    # ("mul_negative_weights", 0.8),
    # ("mul_negative_weights", 0.9),
    # ("opt_debias", "optuna", "CMA-ES", 500),
    # ("opt_debias", "pybobyqa", "Multi-Start BOBYQA", 500),
    ("opt_debias", "nevergrad", "BayesOpt", 200),
    # ("opt_debias", "nevergrad", "Nelder-Mead", 500),
    # ("opt_debias", "nevergrad", "Powell", 500),
)

In [ ]:
unbiased_types = ()
# biased_types = ("perfectly_balanced_redlining", 0.2)
# biased_types = ("balanced_redlining", 0.2)
# biased_types = ("unbalanced_redlining", 0.2)
# biased_types = ("threshold_swapping", 0.2)
biased_types = ("corruption_bias", "CatBoost", 0.001, "gaussian", 1.0)

unadjusted_weights = ()
# adjusted_weights = ("mul_negative_weights", 0.0)
# adjusted_weights = ("opt_debias", "optuna", "CMA-ES", 500)
# adjusted_weights = ("opt_debias", "pybobyqa", "Multi-Start BOBYQA", 500)
adjusted_weights = ("opt_debias", "nevergrad", "BayesOpt", 200)
# adjusted_weights = ("opt_debias", "nevergrad", "Nelder-Mead", 500)
# adjusted_weights = ("opt_debias", "nevergrad", "Powell", 500)


# diff_metric_df_list = []

data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
# data_config_list.append(("muL_best_", "", biased_types, unadjusted_weights))  # Original IRL FE
data_config_list.append(
    ("muL_test_", "", biased_types, adjusted_weights)
)  # Zeroed IRL FE
measurable_metrics = [
    "Acc",
    "AccPar",
    "DemPar",
    "EqOpp",
    "TNRPar",
    # "EqOdds"
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

selected_expert = "OptClfMDPPol"
diff_metric_list = []
dataset_list = []
for dataset in averaged_info:
    dataset_list.append(dataset)
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][selected_algorithm][data_config[2]][
            data_config[3]
        ]
        metric_results = exp_results.loc[measurable_metric]
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[0]
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1).T
diff_metric_df.index = dataset_list
# diff_metric_df_list.append(diff_metric_df)
df1 = diff_metric_df

data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
# data_config_list.append(("muE_", "_mean", biased_types, unadjusted_weights)) # Biased Demo FE
data_config_list.append(
    ("muE_test_unbiased_", "_mean", unbiased_types, unadjusted_weights)
)  # Raw Data FE
measurable_metrics = [
    "Acc",
    "AccPar",
    "DemPar",
    "EqOpp",
    "TNRPar",
    # "EqOdds"
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

selected_expert = "OptClfMDPPol"
diff_metric_list = []
for dataset in averaged_info:
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][selected_algorithm][data_config[2]][
            data_config[3]
        ]
        metric_results = exp_results.loc[measurable_metric]
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[0]
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1).T
diff_metric_df.index = dataset_list
# diff_metric_df_list.append(diff_metric_df)
df2 = diff_metric_df

# -------------------------------------------------------------------
# PLOT CONFIGURATION
# -------------------------------------------------------------------
n_rows = len(df1)  # number of samples per column
n_cols = len(df1.columns)  # number of metrics/columns
colors = {"df1": "tab:blue", "df2": "tab:red"}  # distinct colors
alpha = 0.5  # transparency (0 = fully transparent, 1 = opaque)
bar_width = 0.8  # width of each bar
gap_between_columns = 1.0  # horizontal gap BETWEEN metric groups
label_offset = (
    0.05  # Space between lowest bar bottom and label (as fraction of bar height)
)

fig, ax = plt.subplots(figsize=(18, 7))

# -------------------------------------------------------------------
# PLOT BARS
# -------------------------------------------------------------------
for col_idx, col_name in enumerate(df1.columns):

    # Start position for this metric group on the x‑axis
    start_x = col_idx * (n_rows + gap_between_columns)

    # Extract the values for this column from BOTH DataFrames
    vals_df1 = df1[col_name].values
    vals_df2 = df2[col_name].values

    # Plot every sample (row) in this column
    for row_idx in range(n_rows):
        x_pos = start_x + row_idx

        # Get the row label (index name)
        row_label = df1.index[row_idx]

        # Current values for both bars
        val1 = vals_df1[row_idx]
        val2 = vals_df2[row_idx]

        # -------------------------------------------------------------------
        # CALCULATE SAFE LABEL POSITION (Handles Positive & Negative Data)
        # -------------------------------------------------------------------
        # Find the absolute bottom (most negative value) between the two bars
        lowest_bottom = min(val1, val2, 0.0)

        # Determine offset distance based on bar height to keep proportional spacing
        # Use max absolute height to ensure enough space regardless of bar size
        max_height = max(abs(val1), abs(val2))
        padding_distance = max(0.01 * max_height, 0.01)

        # Calculate final y-position for label (below the lowest bar)
        label_y = lowest_bottom - padding_distance

        # Bar for DataFrame 1
        ax.bar(
            x_pos,
            val1,
            width=bar_width,
            alpha=alpha,
            color=colors["df1"],
            edgecolor="black",
        )

        # Bar for DataFrame 2 (OVERLAPS with DF1 at the SAME x_pos!)
        ax.bar(
            x_pos,
            val2,
            width=bar_width,
            alpha=alpha,
            color=colors["df2"],
            edgecolor="black",
        )

        # Add Row Index Label Below the Bars (Safe for Negative Data)
        ax.text(
            x_pos,
            label_y,
            row_label,
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            rotation=90,
            color="darkgray",
            bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"),
        )

# -------------------------------------------------------------------
# X‑AXIS LABELS (one tick per metric group)
# -------------------------------------------------------------------
# Place a tick at the *center* of each metric group
tick_positions = [
    col_idx * (n_rows + gap_between_columns) + (n_rows - 1) / 2
    for col_idx in range(n_cols)
]
ax.set_xticks(tick_positions)
ax.set_xticklabels(df1.columns, fontsize=11, fontweight="bold", rotation=90)

# -------------------------------------------------------------------
# FINAL TOUCHES
# -------------------------------------------------------------------
ax.set_ylabel("Feature Expectation", fontsize=11)
ax.set_title(
    f"Comparison of How Feature Expectations Changed for {adjusted_weights}",
    fontsize=13,
    fontweight="bold",
)
ax.legend(
    ["Feature Expectation for Adjusted Weights", "Feature Expectation Before Bias Added"],
    loc="upper right",
    framealpha=0.5,
    fancybox=True,
)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(f"./../../experiment_output/fair_irl/exp_plots/Feat_Exp_Before_After.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Visualize what happens to the feature expectations throughout the training process if you fully zero the negative weights for each dataset. Show that if you just zero the negative weights, you get something where the feature expectations dont match what the expert originally was attempting
# feature expectations of the dataset with no bias
# feature expectations of the dataset with bias added
# feature expectations of Fair-IRL trained on bias with no zeroing
# Feature expectations of Fair-IRL trained on bias with zeroing

selected_expert = "OptClfMDPPol"
unbiased_type = ()
# biased_type = ("unbalanced_redlining", 0.2)
# biased_type = ("balanced_redlining", 0.2)
# biased_type = ("perfectly_balanced_redlining", 0.2)
biased_type = ("corruption_bias", "CatBoost", 0.001, "gaussian", 1.0)

all_datasets = [
    "COMPAS",
    # "Boston",
    "Adult",
    "ACSIncome__MA",
    "ACSIncome__MS",
    "ACSIncome__CA",
    "ACSIncome__IL",
    "ACSIncome__AL",
    "ACSIncome__HI",
]

measurable_metrics = [
    "Acc",
    "AccPar",
    "DemPar",
    "EqOpp",
    "TNRPar",
]

muE_test_unbiased_names = ["muE_test_unbiased_" + metric + "_mean" for metric in measurable_metrics]
muE_test_names = ["muE_test_" + metric + "_mean" for metric in measurable_metrics]
muL_test_names = ["muL_test_" + metric for metric in measurable_metrics]

unadjusted_weights = ()
# adjusted_weights = ("mul_negative_weights", 0.0)
adjusted_weights = ("opt_debias", "nevergrad", "BayesOpt", 200)

dataset_results = {}

for dataset in all_datasets:
    muE_test_unbiased = averaged_data[dataset][selected_expert][selected_algorithm][unbiased_type][unadjusted_weights][muE_test_unbiased_names].values.tolist()
    
    muE_test = averaged_data[dataset][selected_expert][selected_algorithm][biased_type][unadjusted_weights][muE_test_names].values.tolist()

    muL_test_unbiased = averaged_data[dataset][selected_expert][selected_algorithm][unbiased_type][unadjusted_weights][muL_test_names].values.tolist()

    muL_test_unadjusted = averaged_data[dataset][selected_expert][selected_algorithm][biased_type][unadjusted_weights][muL_test_names].values.tolist()

    muL_test_zeroed = averaged_data[dataset][selected_expert][selected_algorithm][biased_type][adjusted_weights][muL_test_names].values.tolist()
    
    dataset_results[dataset] = (muE_test_unbiased, muE_test, muL_test_unadjusted, muL_test_zeroed)

# Plotting
for dataset, (muE_test_unbiased, muE_test, muL_test_unadjusted, muL_test_zeroed) in dataset_results.items():
    
    # Bar settings - FIXED: Use 4 bars per cluster with proper positioning
    x = np.arange(len(measurable_metrics))  # x positions for metrics
    width = 0.15  # width of each bar (narrower to fit 4 bars)
    
    fig, ax = plt.subplots(figsize=(16, 6))

    colors_muE_test_unbiased = "red"
    colors_muE_test = "yellow"
    colors_muL_test_unbiased = "orange"
    colors_muL_test_unadjusted = "green"
    colors_muL_test_zeroed = "blue"

    # Clustered bars - FIXED: Proper positioning for 4 bars
    # Calculate offsets so bars are evenly spaced around each x position
    ax.bar(x - 1.5*width, muE_test_unbiased, width, 
           label=f'{selected_expert} Expert on Unbiased Data', color=colors_muE_test_unbiased)
    ax.bar(x - 0.5*width, muE_test, width, 
           label=f'{selected_expert} Expert on Data With {biased_type[0]} Added', color=colors_muE_test)
    ax.bar(x + 0.5*width, muL_test_unbiased, width, 
           label='FairIRL Bias Reduction Model With No Weight Adjustment and No Bias', color=colors_muL_test_unbiased)
    ax.bar(x + 1.5*width, muL_test_unadjusted, width, 
           label=f'FairIRL Bias Reduction Model With No Weight Adjustment on Data With {biased_type[0]} Added', color=colors_muL_test_unadjusted)
    ax.bar(x + 2.5*width, muL_test_zeroed, width, 
           label=f'FairIRL Bias Reduction Model With Weight Adjustment on Data With {biased_type[0]} Added', color=colors_muL_test_zeroed)

    # Labels and title
    ax.set_ylabel('Learned Feature Expectations')
    ax.set_title(f"Learned Feature Expectations Throughout the FairIRL Bias Reduction Training Pipeline for the {dataset} Dataset")
    ax.set_xticks(x)
    ax.set_xticklabels(measurable_metrics, rotation=45, ha='right')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.legend()
    
    # Add grid for better readability
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    
    # Save and show
    plt.savefig(f"./../../experiment_output/fair_irl/exp_plots/Learned_Feat_Exp_of_Train_Pipeline_{dataset}.pdf", bbox_inches="tight")
    plt.show()